# Connect DataCore to Claude Desktop via MCP

Wire DataCore into Claude Desktop using the Model Context Protocol so you can ask plain-English questions like *"What's VNM's trailing P/E versus the banking sector?"* and get answers backed by live Vietnamese market data.

**Why this matters**: every datasource you wire up via MCP becomes available to Claude across every conversation. Once DataCore is connected, you stop copy-pasting CSVs into chats and start asking questions of the data directly.

## What is MCP?

The **Model Context Protocol** (MCP) is an open standard for connecting LLM clients (Claude Desktop, IDE plugins, custom agents) to external data sources and tools. An MCP server exposes a set of tools and resources; the client — in this case Claude Desktop — calls them on demand.

DataCore ships an MCP server (`datacore-mcp`) that exposes:

- `search_datasets(query)` — hybrid search over the catalog
- `get_schema(dataset_id)` — columns, types, coverage
- `query(dataset_id, filters, columns, limit)` — pull rows
- `quota()` — remaining API budget

Claude decides when to call each tool based on the question.

## Prerequisites

1. **Claude Desktop** installed ([download](https://claude.ai/download))
2. **DataCore Python package** installed: `pip install datacore`
3. A valid `DATACORE_API_KEY` in your environment

In [ ]:
import os, sys, json, platform
from pathlib import Path

print('Python:', sys.version.split()[0])
print('Platform:', platform.system())
print('DATACORE_API_KEY set:', bool(os.environ.get('DATACORE_API_KEY')))

## Step 1 — Locate the Claude Desktop config

Claude Desktop reads MCP servers from a JSON file whose location depends on your OS:

| OS      | Path                                                                          |
|---------|-------------------------------------------------------------------------------|
| macOS   | `~/Library/Application Support/Claude/claude_desktop_config.json`             |
| Windows | `%APPDATA%\Claude\claude_desktop_config.json`                                 |
| Linux   | `~/.config/Claude/claude_desktop_config.json`                                 |

In [ ]:
def claude_config_path():
    home = Path.home()
    sysname = platform.system()
    if sysname == 'Darwin':
        return home / 'Library/Application Support/Claude/claude_desktop_config.json'
    if sysname == 'Windows':
        return Path(os.environ.get('APPDATA', home)) / 'Claude/claude_desktop_config.json'
    return home / '.config/Claude/claude_desktop_config.json'

cfg_path = claude_config_path()
print('Config path:', cfg_path)
print('Exists:', cfg_path.exists())

## Step 2 — Write the MCP entry

DataCore ships a helper `write_claude_desktop_config()` that merges an entry into the existing JSON (preserving any other servers you've already configured).

In [ ]:
from datacore import write_claude_desktop_config

result = write_claude_desktop_config(
    api_key=os.environ.get('DATACORE_API_KEY', 'dc_demo_replace_me'),
    server_name='datacore',
    # Optional: pin to a specific Python interpreter
    # python=sys.executable,
)
print('Wrote config to:', result['path'])
print('Entry added under mcpServers.datacore')

## Step 3 — Inspect the JSON that gets written

Useful to know what's on disk — especially if you want to add more servers, change env vars, or commit a sanitized version to your dotfiles.

In [ ]:
# Show the entry that was written. Real keys are redacted in this preview.
preview = {
    "mcpServers": {
        "datacore": {
            "command": sys.executable,
            "args": ["-m", "datacore.mcp"],
            "env": {
                "DATACORE_API_KEY": "dc_live_***redacted***"
            }
        }
    }
}
print(json.dumps(preview, indent=2))

## Step 4 — Restart Claude Desktop

Claude Desktop only reads `claude_desktop_config.json` at startup. Quit it fully and reopen it.

- **macOS**: `Cmd+Q` (closing the window is not enough), then relaunch.
- **Windows**: right-click the tray icon → *Quit*, then relaunch.
- **Linux**: kill the process and `claude-desktop &`.

After it reopens, look for a hammer/tools icon in the chat input area — that's how Claude indicates it can see MCP tools. Click it and you should see `search_datasets`, `get_schema`, `query`, and `quota` listed under **datacore**.

## Step 5 — Example questions to ask Claude

Once the server is live, try these. Claude will pick the right tools to answer each.

In [ ]:
examples = [
    "Which Vietnamese datasets do you have on ESG?",
    "Pull VNM's daily close from Jan 2024 to today and chart the 200-day MA.",
    "Compare trailing P/E across VN30 banks; which is the cheapest?",
    "Build a quick screen: HOSE names with ROE > 18%, P/B < 2, market cap > 10T VND.",
    "What's my DataCore quota right now? Should I cache before the next pull?",
]
for q in examples:
    print(' •', q)

## Step 6 — Troubleshooting

### Tools icon doesn't appear after restart

- Confirm `claude_desktop_config.json` is valid JSON (`python -m json.tool < path`).
- Check Claude Desktop logs:
  - macOS: `~/Library/Logs/Claude/`
  - Windows: `%APPDATA%\Claude\logs\`
- Make sure the Python interpreter under `command` can actually `import datacore.mcp`. If you use pyenv/conda, point `command` at the absolute path to the right Python.

### `RateLimitError` or 401 from inside Claude

- Your `DATACORE_API_KEY` may be invalid or expired. Verify in a notebook:

In [ ]:
from datacore import Client
try:
    Client().whoami()
    print('Key OK')
except Exception as e:
    print('Key problem:', type(e).__name__, e)

### "Server unreachable" / process exits immediately

- Run the server by hand to see the error directly:

```bash
DATACORE_API_KEY=dc_... python -m datacore.mcp
```

It should start and wait on stdin. If it crashes, the traceback you see is the same one Claude swallows.

### Claude says it can't see DataCore tools

- The JSON entry name must match: `mcpServers.datacore`.
- Restart Claude Desktop fully — closing the window isn't enough on macOS.
- If you have other MCP entries, confirm the file isn't malformed (missing comma is the usual culprit).

## Next steps

- `02-research-agent.ipynb` — build an autonomous research agent over DataCore
- `03-rag-on-filings.ipynb` — RAG over HOSE/HNX filings using the same MCP server
- [MCP spec](https://modelcontextprotocol.io) — the protocol DataCore implements